# multi_4h_jma_atr_017

> **Auto-generated documentation header.** Fill the `TODO` fields after review.
> Source notebook: `Notebook1 (18).ipynb`

| Field | Value |
|---|---|
| Section | `strategies` |
| Market | `unknown` |
| Trading style | `unknown` |
| Timeframe | `4h` |
| Code cells | 9 |
| Detected functions | resample_1hr_custom, __init__, compute_atr, compute_jma_and_signals, compute_atr_max, prepare_data, py_backtest_trades, calc_pnl, backtest_trades, calculate_metrics, calculate_composite_score, normalize_metric |

### Research documentation
- **Hypothesis:** _TODO_
- **Alpha source:** _TODO_
- **Indicators / features:** JMA + ATR crossover (detected)
- **Entry logic:** _TODO_
- **Exit logic (SL/TP):** _TODO_
- **Risk management:** _TODO_
- **Assumptions & limitations:** _TODO_
- **Performance (if available):** _TODO_

> Shared utilities live in `src/qresearch/` — prefer importing `jma_signals`, `backtest_trades`,
> `calculate_metrics`, `resample_ohlcv` instead of re-implementing them here.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
data = pd.read_csv('/content/drive/MyDrive/Data/Crypto_2025/Crypto_T5_OCT_2025.csv')
data.rename(columns={
    'Datetime': 'Date',
    'Open': 'open',
    'Close': 'close',
    'Low': 'low',
    'High': 'high',
    'Volume': 'volume'
}, inplace=True)
data

In [ ]:
data['Ticker'].unique()

In [ ]:
# Convert 'Date' column to datetime if it isn't already
data['Date'] = pd.to_datetime(data['Date'])

# Define the date range
start_date = '2025-01-01'
end_date = '2025-06-30'

# Define the tickers you want to filter
selected_tickers = ['SOLUSDT']  # Add tickers you want

# Apply filtering
data = data[
    (data["Date"] >= start_date) &
    (data["Date"] <= end_date) &
    (data["Ticker"].isin(selected_tickers))
]

# Display filtered results
data

In [ ]:
import pandas as pd

# Ensure Date is datetime
data['Date'] = pd.to_datetime(data['Date'])

# Custom resampling function with offset
def resample_1hr_custom(group):
    ticker = group['Ticker'].iloc[0]
    group = group.set_index('Date')
    resampled = group.resample('4h').agg({
        'open': 'first',
        'high': 'max',
        'low': 'min',
        'close': 'last'
    }).dropna().reset_index()
    resampled['Ticker'] = ticker
    return resampled

# Apply per ticker
data_1hr = data.groupby('Ticker').apply(resample_1hr_custom).reset_index(drop=True)

# Reorder columns
data_1hr = data_1hr[['Ticker', 'Date', 'open', 'high', 'low', 'close']]

data = data_1hr
data

In [ ]:
data.dropna()
data

In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm

class ATRIndicator:
    def __init__(self, high_col='high', low_col='low', close_col='close'):
        self.high_col = high_col
        self.low_col = low_col
        self.close_col = close_col

    def compute_atr(self, df, period=14, shift=1):
        high, low, close = df[self.high_col], df[self.low_col], df[self.close_col]
        tr = pd.concat([
            high - low,
            (high - close.shift(1)).abs(),
            (low - close.shift(1)).abs()
        ], axis=1).max(axis=1)
        atr = tr.rolling(window=period, min_periods=1).mean()
        return atr.shift(shift)

def compute_jma_and_signals(df, smooth_length=14, normalization_period=72,
                                upper_threshold=1.75, lower_threshold=-0.75,
                                rs_period=55, rs_percentile=70):
    df = df.copy()
    price = df['close']

    # JMA calculation
    alpha = 2.0 / (smooth_length + 1)
    jma_values = [price.iloc[0]]
    for p in price.iloc[1:]:
        jma_values.append(jma_values[-1] + (p - jma_values[-1]) * alpha)
    df['JMA'] = jma_values

    # Normalization using shifted JMA to avoid look-ahead
    df['JMA_shifted'] = df['JMA'].shift(1)
    df['Normalized_JMA'] = (df['JMA_shifted'] - df['JMA_shifted'].rolling(normalization_period).mean()) / \
                          df['JMA_shifted'].rolling(normalization_period).std()

    # RS Volatility using previous bar's OHLC
    o = df['open'].shift(1).replace(0, np.nan)
    h = df['high'].shift(1)
    l = df['low'].shift(1)
    c = df['close'].shift(1).replace(0, np.nan)

    log_hc, log_ho = np.log(h / c), np.log(h / o)
    log_lc, log_lo = np.log(l / c), np.log(l / o)

    df['RS_Volatility'] = ((log_hc * log_ho + log_lc * log_lo).rolling(window=rs_period).mean()) * 1e6
    df['RS_Threshold'] = df['RS_Volatility'].rolling(normalization_period).quantile(rs_percentile / 100.0)

    # Signals based on shifted/lagged data
    df['Signal'] = 0
    df.loc[(df['Normalized_JMA'] > upper_threshold) & (df['RS_Volatility'] > df['RS_Threshold']), 'Signal'] = 1
    df.loc[(df['Normalized_JMA'] < lower_threshold) & (df['RS_Volatility'] > df['RS_Threshold']), 'Signal'] = 2

    return df

def compute_atr_max(group, multiplier=0.1):
    return pd.concat([
        group['ATR_14'].rolling(20).max() * multiplier,
        group['ATR_5'].rolling(20).max() * multiplier
    ], axis=1).max(axis=1)

def prepare_data(data):
    data['Date'] = pd.to_datetime(data['Date'])

    jma_results = [compute_jma_and_signals(group) for ticker, group in data.groupby('Ticker')]
    data = pd.concat(jma_results, ignore_index=True)

    atr_calc = ATRIndicator()

    for period in [14, 5]:
        col_name = f'ATR_{period}'
        atr_results = [atr_calc.compute_atr(group, period=period, shift=1) for ticker, group in data.groupby('Ticker')]
        data[col_name] = pd.concat(atr_results, ignore_index=True)

    atr_max_results = [compute_atr_max(group) for ticker, group in data.groupby('Ticker')]
    data['ATR_max'] = pd.concat(atr_max_results, ignore_index=True)

    data['Signal'] = data.groupby('Ticker', group_keys=False)['Signal'].shift(1)
    data = data.dropna(subset=['Signal'])
    data['Signal'] = data['Signal'].astype(np.int8)

    return data

import pandas as pd
from typing import List, Tuple, Optional


def py_backtest_trades(
    data,
    ticker: str,
    capital: float,
    fee_rate: float,
    slippage_rate: float,
    entry_sl_pct: float
) -> List[Tuple]:
    """
    Backtest trades with entry SL and trailing stop-loss logic.

    Parameters
    ----------
    data : np.ndarray
        Array with columns [time, open, high, low, close, signal, atr_max].
    ticker : str
        Ticker symbol.
    capital : float
        Initial capital allocated per trade.
    fee_rate : float
        Fee rate applied per trade (as a fraction).
    slippage_rate : float
        Slippage applied to entry (as a fraction).
    entry_sl_pct : float
        Stop-loss percentage for entry candle.

    Returns
    -------
    trades : List[Tuple]
        Each trade tuple contains:
        (ticker, entry_time, exit_time, entry_price, exit_price,
         net_pnl, exit_reason, cumulative_pnl, direction)
    """

    def calc_pnl(entry_price: float, exit_price: float,
                 qty: float, direction: int) -> float:
        """Calculate net PnL after fees."""
        pnl = (exit_price - entry_price) * qty * direction
        fee = fee_rate * (entry_price + exit_price) * qty
        return pnl - fee

    n = data.shape[0]
    trades = []
    cumulative_pnl = 0.0

    # Position state
    position_open = False
    entry_price = sl_price = quantity = 0.0
    entry_time: Optional[pd.Timestamp] = None
    last_exit_time: Optional[pd.Timestamp] = None
    direction = 0

    # Previous candle values
    prev_high = prev_low = prev_close = prev_atr_max = 0.0

    for i in range(n):
        time, open_, high, low, close, signal, atr_max = (
            pd.to_datetime(data[i, 0]),
            float(data[i, 1]), float(data[i, 2]), float(data[i, 3]),
            float(data[i, 4]), int(data[i, 5]), float(data[i, 6])
        )

        # ------------------------
        # ENTRY LOGIC
        # ------------------------
        if not position_open and time != last_exit_time and signal in (1, 2):
            direction = 1 if signal == 1 else -1
            entry_price = open_ * (1 + slippage_rate if direction == 1 else 1 - slippage_rate)
            quantity = capital / entry_price
            entry_time, position_open = time, True

            # Initial SL
            sl_price = entry_price * (1 - entry_sl_pct if direction == 1 else 1 + entry_sl_pct)

            # Check if entry SL is hit on the same candle
            if (direction == 1 and low <= sl_price) or (direction == -1 and high >= sl_price):
                exit_price = sl_price
                net_pnl = calc_pnl(entry_price, exit_price, quantity, direction)
                cumulative_pnl += net_pnl

                trades.append((
                    ticker, entry_time, time, entry_price, exit_price,
                    net_pnl, "Entry SL", cumulative_pnl,
                    "long" if direction == 1 else "short"
                ))

                position_open, last_exit_time = False, time
            continue

        # ------------------------
        # EXIT LOGIC
        # ------------------------
        if position_open and time != entry_time:
            exit_triggered = False
            exit_price = None
            exit_reason = None

            # Update trailing stop based on previous candle
            if i > 0:
                if direction == 1:  # Long
                    sl_price = max(sl_price, prev_close - prev_atr_max)
                else:  # Short
                    sl_price = min(sl_price, prev_close + prev_atr_max)

            # Check trailing SL
            if (direction == 1 and low <= sl_price) or (direction == -1 and high >= sl_price):
                exit_price = sl_price
                exit_reason = "Trailing SL"
                exit_triggered = True

            # Execute exit if triggered
            if exit_triggered:
                net_pnl = calc_pnl(entry_price, exit_price, quantity, direction)
                cumulative_pnl += net_pnl

                trades.append((
                    ticker, entry_time, time, entry_price, exit_price,
                    net_pnl, exit_reason, cumulative_pnl,
                    "long" if direction == 1 else "short"
                ))

                position_open, last_exit_time = False, time

        # ------------------------
        # UPDATE PREVIOUS CANDLE
        # ------------------------
        prev_high, prev_low, prev_close, prev_atr_max = high, low, close, atr_max

    return trades

def backtest_trades(data, initial_capital, fee_rate, slippage_rate, entry_sl_pct):
    data['Date'] = pd.to_datetime(data['Date'])
    data['Date'] = np.array(data['Date'].dt.to_pydatetime())

    tickers = data['Ticker'].unique()
    ticker_map = {
        ticker: data[data['Ticker'] == ticker][["Date", "open", "high", "low", "close", "Signal", "ATR_max"]].to_numpy(dtype=object)
        for ticker in tickers
    }

    all_trades = []
    for ticker in tqdm(tickers, desc="Backtesting"):
        trades = py_backtest_trades(
            ticker_map[ticker],
            ticker,
            initial_capital,
            fee_rate,
            slippage_rate,
            entry_sl_pct
        )
        all_trades.extend(trades)

    return pd.DataFrame(all_trades, columns=[
        "Ticker", "Entry Time", "Exit Time", "Entry Price", "Exit Price",
        "PnL", "Exit Reason", "Cumulative PnL", "Direction"
    ])

# Usage
data = prepare_data(data)  # Assuming 'data' is your input DataFrame
tradebook = backtest_trades(
    data,
    initial_capital=100000,
    fee_rate=0.00025,
    slippage_rate=0.001,
    entry_sl_pct=0.02
)

tradebook.to_csv("/content/drive/MyDrive/tradebook_SOLUSDT_2025.csv", index=False)
tradebook

In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm
from itertools import product
from sklearn.metrics import r2_score
import warnings
warnings.filterwarnings('ignore')

# Import your existing functions
# (ATRIndicator, compute_jma_and_signals, compute_atr_max, prepare_data, py_backtest_trades)

def calculate_metrics(tradebook):
    """Calculate performance metrics from tradebook"""
    if len(tradebook) == 0:
        return {
            'total_trades': 0,
            'profit_to_drawdown': 0,
            'equity_r2': 0,
            'monthly_consistency': 0,
            'daily_drawdown': 0,
            'total_pnl': 0,
            'win_rate': 0,
            'avg_pnl_per_trade': 0
        }

    # Total PnL and trades
    total_pnl = tradebook['PnL'].sum()
    total_trades = len(tradebook)

    # Win rate
    wins = (tradebook['PnL'] > 0).sum()
    win_rate = wins / total_trades if total_trades > 0 else 0

    # Average PnL per trade
    avg_pnl_per_trade = total_pnl / total_trades if total_trades > 0 else 0

    # Equity curve and drawdown
    tradebook = tradebook.sort_values('Exit Time').reset_index(drop=True)
    tradebook['Cumulative PnL'] = tradebook['PnL'].cumsum()

    # Calculate drawdown
    running_max = tradebook['Cumulative PnL'].expanding().max()
    drawdown = running_max - tradebook['Cumulative PnL']
    max_drawdown = drawdown.max()

    # Profit to Drawdown Ratio
    profit_to_drawdown = total_pnl / max_drawdown if max_drawdown > 0 else 0

    # Equity R² (linearity of equity curve)
    if len(tradebook) >= 2:
        x = np.arange(len(tradebook))
        y = tradebook['Cumulative PnL'].values
        try:
            equity_r2 = r2_score(y, np.polyval(np.polyfit(x, y, 1), x))
            equity_r2 = max(0, equity_r2)  # Ensure non-negative
        except:
            equity_r2 = 0
    else:
        equity_r2 = 0

    # Monthly Consistency
    tradebook['Exit Month'] = pd.to_datetime(tradebook['Exit Time']).dt.to_period('M')
    monthly_pnl = tradebook.groupby('Exit Month')['PnL'].sum()

    if len(monthly_pnl) >= 2:
        positive_months = (monthly_pnl > 0).sum()
        monthly_consistency = positive_months / len(monthly_pnl)
    else:
        monthly_consistency = 0

    # Daily Drawdown (average daily drawdown as % of capital)
    tradebook['Exit Date'] = pd.to_datetime(tradebook['Exit Time']).dt.date
    daily_pnl = tradebook.groupby('Exit Date')['PnL'].sum()
    daily_cumsum = daily_pnl.cumsum()
    daily_running_max = daily_cumsum.expanding().max()
    daily_drawdown_series = daily_running_max - daily_cumsum
    avg_daily_drawdown = daily_drawdown_series.mean()

    # Normalize daily drawdown (lower is better)
    # We'll use the ratio of avg daily drawdown to total capital
    initial_capital = 100000  # Assuming from your original code
    daily_drawdown_score = 1 - min(1, avg_daily_drawdown / initial_capital)

    return {
        'total_trades': total_trades,
        'profit_to_drawdown': profit_to_drawdown,
        'equity_r2': equity_r2,
        'monthly_consistency': monthly_consistency,
        'daily_drawdown': daily_drawdown_score,
        'total_pnl': total_pnl,
        'win_rate': win_rate,
        'avg_pnl_per_trade': avg_pnl_per_trade,
        'max_drawdown': max_drawdown
    }

def calculate_composite_score(metrics, weights):
    """
    Calculate weighted composite score
    weights: dict with keys matching metric names
    """
    score = (
        weights['profit_to_drawdown'] * normalize_metric(metrics['profit_to_drawdown'], 'profit_to_drawdown') +
        weights['equity_r2'] * metrics['equity_r2'] +
        weights['monthly_consistency'] * metrics['monthly_consistency'] +
        weights['daily_drawdown'] * metrics['daily_drawdown']
    )
    return score

def normalize_metric(value, metric_name):
    """Normalize metrics to 0-1 range"""
    if metric_name == 'profit_to_drawdown':
        # Typically ranges from -inf to 10+, use sigmoid-like normalization
        return min(1, max(0, value / 5))  # 5 is considered excellent
    return value

def run_single_backtest(data, params, initial_capital, fee_rate, slippage_rate, entry_sl_pct):
    """Run backtest with specific parameters"""
    try:
        # Apply JMA parameters
        jma_results = []
        for ticker, group in data.groupby('Ticker'):
            jma_df = compute_jma_and_signals(
                group,
                smooth_length=params['smooth_length'],
                normalization_period=params['normalization_period'],
                upper_threshold=params['upper_threshold'],
                lower_threshold=params['lower_threshold'],
                rs_period=params['rs_period'],
                rs_percentile=params['rs_percentile']
            )
            jma_results.append(jma_df)

        test_data = pd.concat(jma_results, ignore_index=True)

        # Add ATR calculations
        atr_calc = ATRIndicator()
        for period in [14, 5]:
            col_name = f'ATR_{period}'
            atr_results = [atr_calc.compute_atr(group, period=period, shift=1)
                          for ticker, group in test_data.groupby('Ticker')]
            test_data[col_name] = pd.concat(atr_results, ignore_index=True)

        atr_max_results = [compute_atr_max(group) for ticker, group in test_data.groupby('Ticker')]
        test_data['ATR_max'] = pd.concat(atr_max_results, ignore_index=True)

        # Shift signal and clean data
        test_data['Signal'] = test_data.groupby('Ticker', group_keys=False)['Signal'].shift(1)
        test_data = test_data.dropna(subset=['Signal'])
        test_data['Signal'] = test_data['Signal'].astype(np.int8)

        # Run backtest
        test_data['Date'] = pd.to_datetime(test_data['Date'])
        test_data['Date'] = np.array(test_data['Date'].dt.to_pydatetime())

        tickers = test_data['Ticker'].unique()
        ticker_map = {
            ticker: test_data[test_data['Ticker'] == ticker][
                ["Date", "open", "high", "low", "close", "Signal", "ATR_max"]
            ].to_numpy(dtype=object)
            for ticker in tickers
        }

        all_trades = []
        for ticker in tickers:
            trades = py_backtest_trades(
                ticker_map[ticker], ticker, initial_capital,
                fee_rate, slippage_rate, entry_sl_pct
            )
            all_trades.extend(trades)

        tradebook = pd.DataFrame(all_trades, columns=[
            "Ticker", "Entry Time", "Exit Time", "Entry Price", "Exit Price",
            "PnL", "Exit Reason", "Cumulative PnL", "Direction"
        ])

        return tradebook

    except Exception as e:
        print(f"Error with params {params}: {str(e)}")
        return pd.DataFrame()

def hyperparameter_tuning(data, param_grid, initial_capital=100000,
                         fee_rate=0.00025, slippage_rate=0.0005, entry_sl_pct=0.01):

    # Weights for composite score
    weights = {
        'profit_to_drawdown': 0.1,
        'equity_r2': 0.4,
        'monthly_consistency': 0.4,
        'daily_drawdown': 0.1
    }

    # Generate all parameter combinations
    param_names = list(param_grid.keys())
    param_values = list(param_grid.values())
    combinations = list(product(*param_values))

    print(f"Testing {len(combinations)} parameter combinations...")

    results = []

    for combo in tqdm(combinations, desc="Hyperparameter Tuning"):
        params = dict(zip(param_names, combo))

        # Run backtest
        tradebook = run_single_backtest(data, params, initial_capital,
                                       fee_rate, slippage_rate, entry_sl_pct)

        # Calculate metrics
        metrics = calculate_metrics(tradebook)

        # Calculate composite score
        composite_score = calculate_composite_score(metrics, weights)

        # Store results
        result = {
            **params,
            'composite_score': composite_score,
            'total_trades': metrics['total_trades'],
            'total_pnl': metrics['total_pnl'],
            'profit_to_drawdown': metrics['profit_to_drawdown'],
            'equity_r2': metrics['equity_r2'],
            'monthly_consistency': metrics['monthly_consistency'],
            'daily_drawdown_score': metrics['daily_drawdown'],
            'win_rate': metrics['win_rate'],
            'avg_pnl_per_trade': metrics['avg_pnl_per_trade'],
            'max_drawdown': metrics.get('max_drawdown', 0)
        }
        results.append(result)

    # Convert to DataFrame and sort by composite score
    results_df = pd.DataFrame(results)
    results_df = results_df.sort_values('composite_score', ascending=False).reset_index(drop=True)

    return results_df

# ========================
# USAGE EXAMPLE
# ========================

# Define parameter grid
param_grid = {
    'smooth_length': [14],
    'normalization_period': [36, 48, 60, 72, 84, 96, 108],
    'upper_threshold': [1, 1.25, 1.5, 1.75, 2.0, 2.25, 2.5],
    'lower_threshold': [-1.0, -1.25, -1.5, -1.75, -2.0, -2.25],
    'rs_period': [3, 7, 14, 21, 34, 55],
    'rs_percentile': [10, 30, 50, 70]
}

# Run hyperparameter tuning
results = hyperparameter_tuning(
    data=data,
    param_grid=param_grid,
    initial_capital=100000,
    fee_rate=0.00025,
    slippage_rate=0.001,
    entry_sl_pct=0.01
)

# Save results
results.to_csv('/content/drive/MyDrive/XNGUSD60_hyperparameter_tuning_results.csv', index=False)

# Display top 10 results
print("\nTop 10 Parameter Combinations:")
results.head(20)

# Save top 5 configurations to separate file
results